In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

from orbital_mechanics.solar_system import SolarSystem
from orbital_mechanics.constants import ALTAIRA_MU, AU, YEAR

In [3]:
ss = SolarSystem()

# get id of PlanetX and rogue1
planetX = ss.init_bodies['name'] == 'PlanetX' 
rogue1 = ss.init_bodies['name'] == 'Rogue1'

# spacecraft initial state at t0
import csv
with open('planetx_intercept.csv', 'r', newline='') as csvfile:
    last_state = list(csv.reader(csvfile))[-1]

last_state = np.array(last_state, dtype=np.float64)
t0 = last_state[2]
rv0 = last_state[3:9]
v_inf = last_state[9:12]

In [4]:
# dynamics function
def dydt(t,rv):
    r = rv[0:3]; v = rv[3:6]
    r_mag = np.linalg.norm(r)
    a = -ALTAIRA_MU/r_mag**3 * r
    dr = v; dv = a
    drv = np.concatenate([dr, dv])
    return drv

In [ ]:
# distance from rogue1
def rel_rv_from_rogue1(t,rv):
    r = rv[0:3]; v = rv[3:6]

    # get rogue1's position and velocity
    pl_rv = ss.get_state_at_t(t, rogue1, 1e-12)
    pl_rv = pl_rv[['rx', 'ry', 'rz', 'vx', 'vy', 'vz']].to_numpy(dtype=np.float64).squeeze().ravel()

    pl_r = pl_rv[0:3]
    pl_v = pl_rv[3:6]

    rel_r = r - pl_r
    rel_v = v - pl_v

    return rel_r, rel_v

In [6]:
# event function (check periapsis distance to planetX)
def event(t,rv):
    rel_r, rel_v = rel_rv_from_rogue1(t,rv)
    res = np.dot(rel_r, rel_v)

    return res

event.terminal = True

In [7]:
res = solve_ivp(dydt, [t0, 200*YEAR], rv0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [8]:
vals = []
for i in range(len(res['t'])):
    rel_r, _ = rel_rv_from_rogue1(res['t'][i],res['y'][:,i].ravel())
    rel_r_mag = np.linalg.norm(rel_r)
    dotp = event(res['t'][i], res['y'][:,i].ravel())

    vals.append([res['t'][i], rel_r_mag, dotp])

vals = np.array(vals, dtype=float)
print(vals)

[[ 2.20903200e+09  1.95619397e+10 -1.51410283e+11]
 [ 2.20903200e+09  1.95619397e+10 -1.51410283e+11]
 [ 2.20903200e+09  1.95619397e+10 -1.51410283e+11]
 [ 2.20903204e+09  1.95619394e+10 -1.51410281e+11]
 [ 2.20903235e+09  1.95619370e+10 -1.51410262e+11]
 [ 2.20903554e+09  1.95619123e+10 -1.51410071e+11]
 [ 2.20906741e+09  1.95616657e+10 -1.51408166e+11]
 [ 2.20938608e+09  1.95591991e+10 -1.51389110e+11]
 [ 2.21257279e+09  1.95345338e+10 -1.51198490e+11]
 [ 2.24443991e+09  1.92878811e+10 -1.49286690e+11]
 [ 2.56311114e+09  1.68249608e+10 -1.29685961e+11]
 [ 2.84913664e+09  1.46307082e+10 -1.11571193e+11]
 [ 3.13516215e+09  1.24685938e+10 -9.32709538e+10]
 [ 3.36789826e+09  1.07473939e+10 -7.84477689e+10]
 [ 3.60063437e+09  9.07749923e+09 -6.38451491e+10]
 [ 3.79101888e+09  7.76691243e+09 -5.21540072e+10]
 [ 3.98140340e+09  6.53047249e+09 -4.07510299e+10]
 [ 4.15484243e+09  5.50204445e+09 -3.06429952e+10]
 [ 4.31131354e+09  4.69882088e+09 -2.17581979e+10]
 [ 4.45258311e+09  4.12832714e+

In [9]:
def polar2cart(mag, el, az):
    vx = mag * np.cos(el) * np.cos(az)
    vy = mag * np.cos(el) * np.sin(az)
    vz = mag * np.sin(el)

    dv = np.array([vx, vy, vz])

    return dv

In [10]:
# root func
def rootfunc(y0):
    # calculate delta V
    v0_mag = np.linalg.norm(v_inf)
    dv = polar2cart(v0_mag, y0[0], y0[1])

    # add delta V to planetX's velocity
    pl_rv = ss.get_state_at_t(t0, planetX, 1e-12)
    pl_rv = pl_rv[['rx', 'ry', 'rz', 'vx', 'vy', 'vz']].to_numpy(dtype=np.float64).squeeze().ravel()
    pl_v = pl_rv[3:6]
    v0 = pl_v + dv

    yy0 = np.array([rv0[0], rv0[1], rv0[2], v0[0], v0[1], v0[2]])
    res = solve_ivp(dydt, [t0, 200*YEAR], yy0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)
    rel_r, _ = rel_rv_from_rogue1(res['t'][-1],res['y'][:,-1].ravel())
    print(res['t'][-1] / YEAR, rel_r)
    return rel_r

In [11]:
res = least_squares(rootfunc, [0.0, 0.0], bounds=([-np.pi/2, -np.pi], [np.pi/2, np.pi]))

134.04301939055682 [-1.08360925e+09  2.49267171e+07 -6.19315454e+09]
134.0430196409143 [-1.08360919e+09  2.49266730e+07 -6.19315441e+09]
134.04301972953647 [-1.08360917e+09  2.49268041e+07 -6.19315456e+09]
141.16535075101794 [-4.62186157e+08 -8.04542095e+08 -2.06011328e+09]
141.1653509400001 [-4.62186187e+08 -8.04542130e+08 -2.06011313e+09]
141.16535115214458 [-4.62186083e+08 -8.04542016e+08 -2.06011327e+09]
148.95081139228284 [-2.25949428e+08 -3.19150661e+08 -1.84123219e+08]
148.95081155890404 [-2.25949503e+08 -3.19150708e+08 -1.84123065e+08]
148.95081190551164 [-2.25949375e+08 -3.19150594e+08 -1.84123199e+08]
151.74326316773613 [-19344357.33554649 -22799083.0515396    2695602.86551365]
151.74326332781433 [-19344437.84211731 -22799135.89829916   2695759.62104803]
151.74326374169152 [-19344311.30132103 -22799020.5623374    2695625.50715375]
151.9126826976229 [-58619.72027016 -73341.10738671  -6127.89556077]
151.91268285761188 [-58700.19864273 -73394.29432213  -5970.80379987]
151.912683

In [12]:
res['message']

'`xtol` termination condition is satisfied.'

In [13]:
v1 = v_inf
v2 = polar2cart(np.linalg.norm(v_inf), res['x'][0], res['x'][1])

np.rad2deg(np.arccos(np.dot(v1, v2)/np.linalg.norm(v1)/np.linalg.norm(v2)))

np.float64(17.452396606413398)

In [14]:
def calc_min_max_th():
    rmu = float(ss.init_bodies.loc[rogue1]['mu'].iloc[0])
    rr = float(ss.init_bodies.loc[rogue1]['R'].iloc[0])
    vinf_mag = float(np.linalg.norm(v_inf))
    sindel2 = (rmu/1.1/rr)/(vinf_mag + rmu/1.1/rr)
    delt = 2*np.arcsin(sindel2)
    print(np.rad2deg(delt))

calc_min_max_th()

164.6393903032058


In [15]:
pl_rv = ss.get_state_at_t(t0, planetX, 1e-12)
pl_rv = pl_rv[['rx', 'ry', 'rz', 'vx', 'vy', 'vz']].to_numpy(dtype=np.float64).squeeze().ravel()
pl_v = pl_rv[3:6]
v2_i = pl_v + v2

In [16]:
yy0 = np.array([rv0[0], rv0[1], rv0[2], v2_i[0], v2_i[1], v2_i[2]])
res = solve_ivp(dydt, [t0, 200*YEAR], yy0, events=event, method='DOP853', rtol=1e-12, atol=1e-12)

In [17]:
res['t'] / YEAR

array([ 70.        ,  70.00000001,  70.0000001 ,  70.00000103,
        70.0000103 ,  70.00010301,  70.00103011,  70.0103011 ,
        70.10301097,  71.03010972,  80.30109719,  92.23791268,
       101.75682614, 110.29925763, 117.95834571, 124.82728719,
       130.98950199, 136.52073311, 141.49059689, 146.46046066,
       150.45689501, 151.91328711])

In [18]:
(res['y'][0:3]).T / AU

array([[-138.58320844,   10.67495418,  -28.02214553],
       [-138.58320844,   10.67495418,  -28.02214553],
       [-138.58320836,   10.67495417,  -28.0221455 ],
       [-138.58320761,   10.6749541 ,  -28.02214522],
       [-138.58320006,   10.6749534 ,  -28.02214239],
       [-138.58312462,   10.67494645,  -28.02211409],
       [-138.58237023,   10.67487689,  -28.02183107],
       [-138.57482622,   10.67418131,  -28.01900086],
       [-138.49937654,   10.66722479,  -27.99069681],
       [-137.74392269,   10.59758589,  -27.70746322],
       [-130.0893817 ,    9.89352948,  -24.8553131 ],
       [-119.93805669,    8.96457046,  -21.12696937],
       [-111.56653384,    8.20310583,  -18.10499722],
       [-103.80920501,    7.50177165,  -15.35334198],
       [ -96.62529359,    6.85643024,  -12.8523974 ],
       [ -89.96881974,    6.2625023 ,  -10.58104018],
       [ -83.79769058,    5.71579388,   -8.51991253],
       [ -78.07219491,    5.21236118,   -6.65089841],
       [ -72.75378734,    4.

In [19]:
rel_r, rel_v = rel_rv_from_rogue1(res['t'][-1], res['y'][:,-1].T)

In [20]:
rel_r, rel_v

(array([ 1.90734863e-06, -1.19209290e-07,  1.22189522e-06]),
 array([ 5.54587253, -4.54156022,  1.30348063]))

In [21]:
planetx_id = float(ss.init_bodies.loc[planetX]['id'].to_numpy().squeeze())
rogue1_id = float(ss.init_bodies.loc[rogue1]['id'].to_numpy().squeeze())

# exit from the flyby:
sol_line_1_raw = [
    planetx_id, '1', t0, rv0[0:3], v2_i, v2
]

# conic flyby
sol_line_2_raw = [
    '0', '0', t0, rv0[0:3], v2_i, [0.0, 0.0, 0.0]
]

sol_line_3_raw = [
    '0', '0', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, [0.0, 0.0, 0.0]
]

# encounter with rogue1
sol_line_4_raw = [
    rogue1_id, '1', res['t'][-1], res['y'][0:3,-1].T, res['y'][3:6,-1].T, rel_v
]

In [22]:
def flatten_list(l):
    out = []
    for v in l:
        if isinstance(v, (int, float, str)):
            out.append(float(v))
        elif isinstance(v, (list, np.ndarray)):
            for y in v:
                out.append(float(y))
    return out

In [23]:
sol_lines = list()
sol_lines.append(flatten_list(sol_line_1_raw))
sol_lines.append(flatten_list(sol_line_2_raw))
sol_lines.append(flatten_list(sol_line_3_raw))
sol_lines.append(flatten_list(sol_line_4_raw))

In [24]:
sol_lines

[[10.0,
  1.0,
  2209032000.0,
  -20731752896.68302,
  1596950414.5640807,
  -4192053304.179736,
  3.857374958498649,
  -0.35566128955903453,
  1.4471417684709837,
  3.9485112737171058,
  1.2726238199140607,
  2.768303784879681],
 [0.0,
  0.0,
  2209032000.0,
  -20731752896.68302,
  1596950414.5640807,
  -4192053304.179736,
  3.857374958498649,
  -0.35566128955903453,
  1.4471417684709837,
  0.0,
  0.0,
  0.0],
 [0.0,
  0.0,
  4794018749.3091,
  -9116179240.16677,
  558404344.795056,
  -204139662.9522742,
  5.616122239585088,
  -0.47712011532975673,
  1.6430060244181655,
  0.0,
  0.0,
  0.0],
 [9.0,
  1.0,
  4794018749.3091,
  -9116179240.16677,
  558404344.795056,
  -204139662.9522742,
  5.616122239585088,
  -0.47712011532975673,
  1.6430060244181655,
  5.545872529789342,
  -4.541560221212328,
  1.3034806253290037]]

In [25]:
import shutil
import csv

shutil.copyfile('planetx_intercept.csv', 'rogue1_intercept.csv')

with open('rogue1_intercept.csv', 'a', newline='') as f:
    # header = ['#body_id','flag','epoch','rx','ry','rz','vx','vy','vz','ux','uy','uz']
    writer = csv.writer(f)
    # writer.writerow(header)
    for l in sol_lines:
        writer.writerow(l)